***

Preparing Workspace

***

In [ ]:


## Packages ---

import numpy as np
import pandas as pd
import getpass
from pathlib import Path
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft

# Plotting
import matplotlib.pyplot as plt 
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio



## File paths ---

export=False

user = getpass.getuser()
path_users = Path.home()

path_sp = path_users / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents'
path_raw = path_sp / 'Process Revamp' / 'Task 8. Reproduce Progress Report indicators' / 'Indicator Data' / 'SACOG Data'
path_main = path_sp / 'Data'
path_prod = path_sp / 'Products'
path_git = path_users / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen'
path_code    = path_git / 'Data' / 'Census'
path_config0 = path_git / 'config'
path_config  = path_code / 'config'
path_server = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data")

path_housing = path_main / 'Vibrant and Inclusive Places' / 'Development' / 'SACOG Housing Dataset'
file_about = path_sp / 'Process Revamp' / 'Task 6. Process Map' / 'About Indicators.xlsx'


## User defined functions ---


path_func = path_config0 / 'Functions.py'

with path_func.open("r") as f:
    exec(f.read())


def export_housing(df):
    workbook_name = f'{indicator_name} {geography} {source}.xlsx'
    path_out_workbook = path_out / workbook_name
    df_about = write_about(sample_type      = sample_type
                           , indicator_name = indicator_name
                           , year_start     = year_start
                           , year_end       = year_end
                           , path_config0   = path_config0
                           , geography      = geography)
    with pd.ExcelWriter(path_out_workbook, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_about.to_excel(writer, sheet_name='About'  , index=False, header=False)
        df      .to_excel(writer, sheet_name=geography, index=False,             )



***

DOF Summarized Data

***

In [ ]:


## Importing ---

year_start = 2001
year_end   = 2024

years_to_import = range(year_start, year_end+1)

list_df = []

for year in tqdm(years_to_import):
    file_housing = path_housing / 'DOF_Summarized_00_24_detailed_State.xlsx' # Developed by Warren
    df_year = pd.read_excel(file_housing, sheet_name = str(year))
    df_year['Year'] = year        
    df_year = df_year.rename(columns = {'City':'Jurisdiction'})
    list_df.append(df_year)

df_housing = pd.concat(list_df)
df_housing = df_housing.set_index(['MPO', 'County', 'Jurisdiction', 'Year']).reset_index()
df_housing.loc[df_housing['Jurisdiction'].str.contains('COUNTY'), 'Jurisdiction'] = 'UNINCORPORATED'
df_housing = df_housing[df_housing['County'] != 'Region Total']
df_housing = df_housing[df_housing['County'] != 'MPO Total']

df_housing = df_housing.set_index(['MPO', 'County', 'Jurisdiction', 'Year']).reset_index()
df_housing = df_housing.sort_values(['MPO', 'County', 'Jurisdiction', 'Year'], ascending = [True, True, True, True])
df_housing = df_housing.drop(['Single', 'Multiple', 'Mobile Homes'], axis = 1)
df_housing.loc[df_housing['MPO'] == 'San Joaquin Valley', 'MPO'] = 'CVCOG'
df_housing.head()



***

Production_2

***

In [ ]:


metrics = ['Total_SF_MF', 'SF', 'MF']

# # Jurisdiction level
# print('Organizing indicator Production_2 by Jurisdictions')
# gb_jurisdiction = df_housing.groupby(['MPO', 'County', 'Jurisdiction', 'Year'], sort=False, as_index=False)
# df_prod2_a = gb_jurisdiction[metrics].sum()
# df_prod2_a2 = df_prod2_a.copy()
# df_prod2_a2[metrics] = df_prod2_a2.groupby(['MPO', 'County', 'Jurisdiction'])[metrics].diff()
# df_prod2_a2 = df_prod2_a2.rename(columns = {'Total_SF_MF':'Total_SF_MF_diff', 'SF':'SF_diff', 'MF':'MF_diff'})
# df_prod2_a = df_prod2_a.merge(df_prod2_a2, on = ['MPO', 'County', 'Jurisdiction', 'Year'], how = 'left')
# df_prod2_a = df_prod2_a.sort_values(['MPO', 'County', 'Jurisdiction', 'Year'], ascending = [True, True, True, False])
# df_prod2_a = df_prod2_a.reset_index(drop = True)
# display(df_prod2_a.head(5))


# County level 
print('Organizing indicator Production_2 by Counties')
df_housing2 = df_housing[(df_housing['MPO'] == 'SACOG') & (df_housing['Jurisdiction'] == 'County Total')]
gb_county = df_housing2.groupby(['MPO', 'County', 'Year'], sort=False, as_index=False)
df_prod2_b = gb_county[metrics].sum()
df_prod2_b2 = df_prod2_b.copy()
df_prod2_b2[metrics] = df_prod2_b2.groupby(['MPO', 'County'])[metrics].diff()
df_prod2_b2 = df_prod2_b2.rename(columns = {'Total_SF_MF':'Total_SF_MF_diff', 'SF':'SF_diff', 'MF':'MF_diff'})
df_prod2_b = df_prod2_b.merge(df_prod2_b2, on = ['MPO', 'County', 'Year'], how = 'left')
df_prod2_b = df_prod2_b.sort_values(['MPO', 'County', 'Year'], ascending = [True, True, False])
df_prod2_b = df_prod2_b.reset_index(drop = True)
df_prod2_b['SF_diff_pct'] = df_prod2_b['SF_diff']/df_prod2_b['Total_SF_MF_diff']*100
df_prod2_b['MF_diff_pct'] = df_prod2_b['MF_diff']/df_prod2_b['Total_SF_MF_diff']*100

df_prod2_b.loc[df_prod2_b['SF_diff_pct'] > 100, 'SF_diff_pct'] = 100
df_prod2_b.loc[df_prod2_b['SF_diff_pct'] <   0, 'SF_diff_pct'] =   0
df_prod2_b.loc[df_prod2_b['MF_diff_pct'] > 100, 'MF_diff_pct'] = 100
df_prod2_b.loc[df_prod2_b['MF_diff_pct'] <   0, 'MF_diff_pct'] =   0

display(df_prod2_b.head(5))

# MPO level
print('Organizing indicator Production_2 by MPO')
df_housing2 = df_housing[df_housing['Jurisdiction'] == 'County Total']
gb_mpo = df_housing2.groupby(['MPO', 'Year'], sort=False, as_index=False)
df_prod2_c = gb_mpo[metrics].sum()
df_prod2_c2 = df_prod2_c.copy()
df_prod2_c2[metrics] = df_prod2_c2.groupby(['MPO'])[metrics].diff()
df_prod2_c2 = df_prod2_c2.rename(columns = {'Total_SF_MF':'Total_SF_MF_diff', 'SF':'SF_diff', 'MF':'MF_diff'})
df_prod2_c = df_prod2_c.merge(df_prod2_c2, on = ['MPO', 'Year'], how = 'left')
df_prod2_c = df_prod2_c.sort_values(['MPO', 'Year'], ascending = [True, False])
df_prod2_c = df_prod2_c.reset_index(drop = True)
df_prod2_c['SF_diff_pct'] = df_prod2_c['SF_diff']/df_prod2_c['Total_SF_MF_diff']*100
df_prod2_c['MF_diff_pct'] = df_prod2_c['MF_diff']/df_prod2_c['Total_SF_MF_diff']*100

df_prod2_c.loc[df_prod2_c['SF_diff_pct'] > 100, 'SF_diff_pct'] = 100
df_prod2_c.loc[df_prod2_c['SF_diff_pct'] <   0, 'SF_diff_pct'] =   0
df_prod2_c.loc[df_prod2_c['MF_diff_pct'] > 100, 'MF_diff_pct'] = 100
df_prod2_c.loc[df_prod2_c['MF_diff_pct'] <   0, 'MF_diff_pct'] =   0

display(df_prod2_c.head(5))



In [ ]:


# df_plot1 = df_prod2_a.copy()
# df_plot2 = df_prod2_a.copy()
# df_plot3 = df_prod2_a.copy()


# df_plot1 = pd.melt(df_plot1, id_vars = ['Year', 'County', 'Jurisdiction'])
# df_plot1 = df_plot1[df_plot1['variable'] == 'SF_diff']
# fig = px.line(df_plot1, x='Year', y='value', color='County', line_dash = 'Jurisdiction', markers=True)
# fig.update_layout(title = 'SF Units by Jurisdiction')
# # fig.write_html(os.path.join(path_out, indicator_name, 'plots', 'Jurisdictions', indicator_name + '_SF Units Added by Jurisdiction_line.html'))
# fig.show()

# df_plot2 = pd.melt(df_plot2, id_vars = ['Year', 'County', 'Jurisdiction'])
# df_plot2 = df_plot2[df_plot2['variable'] == 'MF_diff']
# fig = px.line(df_plot2, x='Year', y='value', color='County', line_dash = 'Jurisdiction', markers=True)
# fig.update_layout(title = 'MF Units by Jurisdiction')
# # fig.write_html(os.path.join(path_out, indicator_name, 'plots', 'Jurisdictions', indicator_name + '_MF Units Added by Jurisdiction_line.html'))
# fig.show()

# df_plot3 = pd.melt(df_plot3, id_vars = ['Year', 'County', 'Jurisdiction'])
# df_plot3 = df_plot3[df_plot3['variable'] == 'Total_SF_MF_diff']
# fig = px.line(df_plot3, x='Year', y='value', color='County', line_dash = 'Jurisdiction', markers=True)
# fig.update_layout(title = 'Total SF and MF Units by Jurisdiction')
# # fig.write_html(os.path.join(path_out, indicator_name, 'plots', 'Jurisdictions', indicator_name + '_Total Units Added by Jurisdiction_line.html'))
# fig.show()


df_plot1 = df_prod2_b.copy()
df_plot2 = df_prod2_b.copy()
df_plot3 = df_prod2_b.copy()

df_plot1 = pd.melt(df_plot1, id_vars = ['Year', 'County'])
df_plot1 = df_plot1[df_plot1['variable'].isin(['SF_diff', 'MF_diff'])]
fig = px.line(df_plot1, x='Year', y='value', color='County', line_dash = 'variable', markers=True)
fig.update_layout(title = 'MF vs SF Units by County')
fig.show()

df_plot2 = pd.melt(df_plot2, id_vars = ['Year', 'County'])
df_plot2 = df_plot2[df_plot2['variable'].isin(['SF_diff_pct', 'MF_diff_pct'])]
fig = px.line(df_plot2, x='Year', y='value', color='County', line_dash = 'variable', markers=True)
fig.update_layout(title = 'MF vs SF Units by County (%)')
fig.show()

df_plot3 = pd.melt(df_plot3, id_vars = ['Year', 'County'])
df_plot3 = df_plot3[df_plot3['variable'].isin(['Total_SF_MF_diff'])]
fig = px.line(df_plot3, x='Year', y='value', color='County', markers=True)
fig.update_layout(title = 'Total SF and MF Units by County')
fig.show()

df_plot1 = df_prod2_c.copy()
df_plot2 = df_prod2_c.copy()

df_plot1 = pd.melt(df_plot1, id_vars = ['MPO', 'Year'])
df_plot1 = df_plot1[df_plot1['variable'].isin(['SF_diff', 'MF_diff'])]
fig = px.line(df_plot1, x='Year', y='value', color='MPO', line_dash = 'variable', markers=True)
fig.update_layout(title = 'MF vs SF Units SACOG')
fig.show()

df_plot2 = pd.melt(df_plot2, id_vars = ['MPO', 'Year'])
df_plot2 = df_plot2[df_plot2['variable'].isin(['SF_diff_pct', 'MF_diff_pct'])]
fig = px.line(df_plot2, x='Year', y='value', color='MPO', line_dash = 'variable', markers=True)
fig.update_layout(title = 'MF vs SF Units SACOG (%)')
fig.show()



In [ ]:


if export:
    indicator_name = 'Production_2'
    source = 'DOF Summarized Data'
    sample_type = 'SACOG Housing'

    # Update overall about documentations workbook
    df_about = write_about(sample_type   = sample_type
                        , indicator_name = indicator_name
                        , year_start     = year_start
                        , year_end       = year_end
                        , path_config0   = path_config0)
    with pd.ExcelWriter(file_about, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_about.to_excel(writer, index=False, sheet_name=indicator_name, header=False)


    # Exporting
    print('Exporting...');print('')

    path_out_sp = path_main / 'Vibrant and Inclusive Places' / 'Development' / 'Housing Production' / indicator_name
    paths_out = [path_out_sp, path_server]

    for path_out in paths_out:
        
        # geography = 'Jurisdictions'
        # export_housing(df_prod2_a)

        geography = 'Counties'
        export_housing(df_prod2_b)
        
        geography = 'MPO'
        export_housing(df_prod2_c)

    print('Success!!')

    

***

Production_3

***

In [ ]:
# Importing Datasets
pop_5_path = path_main / 'Vibrant and Inclusive Places' / 'People and Community' / 'Pop and Demographics'
file_pop_5 = pop_5_path / 'DOF_E5_and_E8_Jurisdictions.xlsx'
df_pop_5 = pd.read_excel(file_pop_5)
df_pop_5 = df_pop_5.sort_values(['County', 'Jurisdiction', 'Year'], ascending=[True,True,True])


gb_jurisdiction = df_housing.groupby(['MPO', 'County', 'Jurisdiction', 'Year'], sort=False, as_index=False)
gb_county       = df_housing.groupby(['MPO', 'County'                , 'Year'], sort=False, as_index=False)
gb_mpo          = df_housing.groupby(['MPO'                          , 'Year'], sort=False, as_index=False)

metrics = ['Total_SF_MF']

# # Jurisdiction level
# print('Organizing indicator Production_3 by Jurisdictions')
# df_prod3_a = gb_jurisdiction[metrics].sum()
# df_prod3_a2 = df_prod3_a.copy()
# df_prod3_a2[metrics] = df_prod3_a2.groupby(['MPO', 'County', 'Jurisdiction'])[metrics].diff()
# df_prod3_a2 = df_prod3_a2.rename(columns = {'Total_SF_MF':'Total_SF_MF_diff'})
# df_prod3_a = df_prod3_a.merge(df_prod3_a2, on = ['MPO', 'County', 'Jurisdiction', 'Year'], how = 'left')
# df_prod3_a = df_prod3_a.sort_values(['MPO', 'County', 'Jurisdiction', 'Year'], ascending = [True, True, True, False])
# df_prod3_a = df_prod3_a.reset_index(drop = True)

# df_pop5_a = df_pop_5.groupby(['MPO', 'County', 'Jurisdiction', 'Year'], as_index = False)['Population'].sum()
# df_prod3_a = df_prod3_a.merge(df_pop5_a, on = ['MPO', 'County', 'Jurisdiction', 'Year'], how = 'left')
# df_prod3_a['Total_SF_MF_diff_per_1000_pop'] = df_prod3_a['Total_SF_MF_diff']/df_prod3_a['Population']*1000
# display(df_prod3_a.head(5))



# County level
print('Organizing indicator Production_3 by Counties')
df_housing2 = df_housing[(df_housing['MPO'] == 'SACOG') & (df_housing['Jurisdiction'] == 'County Total')]
gb_county = df_housing2.groupby(['MPO', 'County', 'Year'], sort=False, as_index=False)
df_prod3_b = gb_county[metrics].sum()
df_prod3_b2 = df_prod3_b.copy()
df_prod3_b2[metrics] = df_prod3_b2.groupby(['MPO', 'County'])[metrics].diff()
df_prod3_b2 = df_prod3_b2.rename(columns = {'Total_SF_MF':'Total_SF_MF_diff'})
df_prod3_b = df_prod3_b.merge(df_prod3_b2, on = ['MPO', 'County', 'Year'], how = 'left')
df_prod3_b = df_prod3_b.sort_values(['MPO', 'County', 'Year'], ascending = [True, True, False])
df_prod3_b = df_prod3_b.reset_index(drop = True)

df_pop5_b = df_pop_5.groupby(['MPO', 'County', 'Year'], as_index = False)['Population'].sum()
df_prod3_b = df_prod3_b.merge(df_pop5_b, on = ['MPO', 'County', 'Year'], how = 'left')
df_prod3_b['Total_SF_MF_diff_per_1000_pop'] = df_prod3_b['Total_SF_MF_diff']/df_prod3_b['Population']*1000
display(df_prod3_b.head(5))



# Jurisdiction level
print('Organizing indicator Production_3 by MPO')
df_housing2 = df_housing[(df_housing['Jurisdiction'] == 'County Total') | (df_housing['MPO'] == 'State Total')]
gb_mpo = df_housing2.groupby(['MPO', 'Year'], sort=False, as_index=False)
df_prod3_c = gb_mpo[metrics].sum()
df_prod3_c2 = df_prod3_c.copy()
df_prod3_c2[metrics] = df_prod3_c2.groupby(['MPO'])[metrics].diff()
df_prod3_c2 = df_prod3_c2.rename(columns = {'Total_SF_MF':'Total_SF_MF_diff'})
df_prod3_c = df_prod3_c.merge(df_prod3_c2, on = ['MPO', 'Year'], how = 'left')
df_prod3_c = df_prod3_c.sort_values(['MPO', 'Year'], ascending = [True, False])
df_prod3_c = df_prod3_c.reset_index(drop = True)

df_pop5_c = df_pop_5.groupby(['MPO', 'Year'], as_index = False)['Population'].sum()
df_pop_5_CA = df_pop_5.groupby(['Year'], as_index = False)['Population'].sum()
df_pop_5_CA['MPO'] = 'State Total'
df_pop5_c = pd.concat([df_pop5_c, df_pop_5_CA])
df_prod3_c = df_prod3_c.merge(df_pop5_c, on = ['MPO', 'Year'], how = 'left')
df_prod3_c['Total_SF_MF_diff_per_1000_pop'] = df_prod3_c['Total_SF_MF_diff']/df_prod3_c['Population']*1000
display(df_prod3_c.head(5))



In [ ]:
df_plot1 = df_prod3_c.copy()

df_plot1 = pd.melt(df_plot1, id_vars = ['MPO', 'Year'])
df_plot1 = df_plot1[df_plot1['variable'].isin(['Total_SF_MF_diff_per_1000_pop'])]
fig = px.line(df_plot1, x='Year', y='value', color='MPO', line_dash = 'variable', markers=True)
fig.update_layout(title = 'Total SF and MF Units Growth per 1k People by MPO')
fig.show()


In [ ]:
conditions = [
                df_plot1['MPO'] == 'CVCOG'
                , df_plot1['MPO'] == 'MTC'
                , df_plot1['MPO'] == 'Rest of CA'
                , df_plot1['MPO'] == 'SACOG'
                , df_plot1['MPO'] == 'SANDAG'
                , df_plot1['MPO'] == 'SCAG'
                , df_plot1['MPO'] == 'State Total'
            ]
# choices = ['San Joaquin Valley (7-County)'
#            , 'San Francisco Bay Area (MTC, 9-County)'
#            , 'Rest of State'
#            , 'Sacramento (SACOG, 6-County)'
#            , 'San Diego (SANDAG, 1-County)'
#            , 'Los Angeles (SCAG, 9-County)'
#            , 'State Total']

choices = ['San Joaquin Valley'
           , 'San Francisco Bay Area'
           , 'Rest of State'
           , 'Sacramento'
           , 'San Diego'
           , 'Los Angeles'
           , 'State Total']

df_plot1['MPO'] = np.select(conditions, choices, default = 'No')
df_plot1.MPO.unique()

display(df_plot1)

# df_plot1.to_csv(os.path.join(path_agol, indicator_name, indicator_name + '_SACOG_DOF_Summarized_Data.csv'), index = False)


In [ ]:


if export:

    indicator_name = 'Production_3'
    source = 'DOF Summarized Data'
    sample_type = 'SACOG Housing'

    # Update overall about documentations workbook
    df_about = write_about(sample_type      = sample_type
                        , indicator_name = indicator_name
                        , year_start     = year_start
                        , year_end       = year_end
                        , path_config0   = path_config0)
    with pd.ExcelWriter(file_about, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_about.to_excel(writer, index=False, sheet_name=indicator_name, header=False)


    # Exporting
    print('Exporting...');print('')
    path_out_sp = path_main / 'Vibrant and Inclusive Places' / 'Development' / 'Housing Production' / indicator_name
    paths_out = [path_out_sp, path_server]

    for path_out in paths_out:
        
        # geography = 'Jurisdictions'
        # export_housing(df_prod2_a)
            
        geography = 'Counties'
        export_housing(df_prod3_b)
        
        geography = 'MPO'
        export_housing(df_prod3_c)

    print('Success!!')

    